[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_3_Nonlinear_Charpit_AR.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs_ar/Lab_3_Nonlinear_Charpit_AR.ipynb)


In [ ]:
# 🚀 تهيئة مسار المشروع تلقائياً (يعمل على JupyterLab المحلي، وBinder، وGoogle Colab)
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
except NameError:
    pass

# التأكد من إضافة المجلد الرئيسي للمشروع (الذي يحتوي على labs/ و rover_path_finder/) إلى مسار بايثون
# سواء تم تشغيل النوت بوك من المجلد الرئيسي أو من داخل labs_ar/ أو solutions/labs_ar/
_cwd = os.path.abspath(os.getcwd())
for _candidate in [_cwd, os.path.abspath(os.path.join(_cwd, '..')), os.path.abspath(os.path.join(_cwd, '..', '..'))]:
    if os.path.isdir(os.path.join(_candidate, 'labs', 'tests')) and os.path.isdir(os.path.join(_candidate, 'rover_path_finder')):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break
print('[OK] تم تهيئة مسار المشروع بنجاح!')


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

# ‏📡 اللاب 3: المعادلات غير الخطية من الرتبة الأولى، طريقة شاربيت، وهوامش أمان العوائق

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏<i>غرفة التحكم: هامش الأمان الإقليدي حول الصخور ومعادلة المسافة (Eikonal)</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المقرر:</b> المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المرجع في المذكرة:</b> الباب الأول، الأقسام 1.3.2–1.3.6 — <b>ترقيم المذكرة ص 16–26</b> <i>(صفحات ملف الـ PDF من 17 إلى 27)</i>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🗺️ موقعك في سلسلة بناء محرك الملاحة (Rover Pipeline)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; background-color:rgba(128,128,128,0.12); padding:10px 14px; border-radius:6px; overflow-x:auto; margin:10px 0; font-family:monospace; white-space:pre;">‏[اللاب 1: حساب الميول] ──► [اللاب 2: خطوط الانسياب] ──► 📍 [اللاب 3: معادلة المسافة وهوامش الأمان] ──► [اللاب 4–5: نظرية لابلاس] ──► [اللاب 6: محرك الملاحة الكامل]</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 حجر رشيد (The Rosetta Stone): الربط بين السبورة ومحرك المركبة

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏1. رمز المذكرة (PDF pp. 17–27)

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏2. المعنى الهندسي والفيزيائي

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏3. الدالة الفعلية في المشروع (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/</code>‏)

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(x, y, z, p, q) = 0}$</span>‏ غير خطية

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏معادلة تفاضلية تحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p^2, q^2}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${pq}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏حساب المسافات الإقليدية وجبهات الموجات

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p^2 + q^2 = 1}$</span>‏ (أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\Vert \nabla z\Vert  = 1}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>معادلة المسافة (Eikonal):</b> دالة تزداد بمعدل 1 لكل متر نبتعد فيه عن المصدر

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate(raw, rover&#95;radius)</code>‏</a> (توسيع العوائق بنصف قطر المركبة)

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z = px + qy + f(p, q)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>صيغة كليروت (Clairaut Form):</b> عائلة مستويات مماسة؛ حلها التام ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z = ax + by + f(a,b)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تمثيل جبهات الموجات المستوية وزوايا المسح

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(x - x_0)^2 + (y - y_0)^2 \le R^2}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏قرص الأمان الدائري حول كل صخرة أو حفرة لحماية هيكل المركبة

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">yy&#42;&#42;2 + xx&#42;&#42;2 &lt;= radius&#42;&#42;2</code>‏ في ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate</code>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏1. الفكرة ببساطة: كيف تحسب المركبة المسافة الآمنة حول الصخور؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏المركبة الفضائية ليست نقطة هندسية بلا أبعاد، بل لها عرض فعلي (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;radius</code>‏). إذا خططنا مساراً يمر بمحاذاة حافة حفرة مباشرة، فستسقط عجلات المركبة في الحفرة!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لذلك، قبل حساب حقل الجهد في <a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.py</code>‏</a>، تقوم الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">dilate(raw, radius)</code>‏ بحساب <b>خريطة المسافة الإقليدية</b> حول كل عائق لتوسيعه بمقدار نصف قطر المركبة:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• إذا سمينا المسافة الإقليدية من نقطة الأصل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z(x,y) = \sqrt{x^2 + y^2}}$</span>‏، فإن ميل هذه الدالة في اتجاه ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ هو:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p = \frac{\partial z}{\partial x} = \frac{x}{\sqrt{x^2+y^2}}, \qquad q = \frac{\partial z}{\partial y} = \frac{y}{\sqrt{x^2+y^2}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• بتربيع الطرفين وجمعهما نحصل على أشهر معادلة تفاضلية جزئية غير خطية من الرتبة الأولى (<b>معادلة Eikonal</b>):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p^2 + q^2 = 1 \quad \Longleftrightarrow \quad \|\nabla z\| = 1$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏هذه المعادلة <b>غير خطية</b> لأن المشتقتين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏ مرفوعتان للأس 2، فلا يمكن حلها بطريقة لاجرانج التي درسناها في اللاب 2! في هذا اللاب سنتعلم كيف نحل المعادلات غير الخطية من الرتبة الأولى باستخدام <b>الأشكال القياسية الأربعة</b> و<b>طريقة شاربيت العامة (Charpit's Method)</b>.

</div>

</div>


In [ ]:
import os, sys
for _p in ['.', '..', '../..']:
    _ap = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_ap, 'labs', 'tests')) and _ap not in sys.path:
        sys.path.insert(0, _ap)
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab3

%matplotlib inline
print("تم تهيئة بيئة اللاب 3 بنجاح!")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏2. الشرح الكامل للأشكال القياسية الأربعة وطريقة شاربيت العامة (PDF ص 17–27)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما تواجه معادلة غير خطية من الرتبة الأولى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y, z, p, q) = 0}$</span>‏، هدفنا هو إيجاد <b>التكامل التام (Complete Integral)</b>، وهو حل يحتوي على <b>ثابتين اختياريين</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ (أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a, c}$</span>‏) بعدد المتغيرات المستقلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x, y)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 خريطة القرار السريع: كيف تختار طريقة الحل في 5 ثوانٍ؟

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏الشكل القياسي في المذكرة

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏كيف تميزه بمجرد النظر؟

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏خطوات الحل القياسية

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>Form 1 (القسم 1.3.2)</b> <i>(PDF ص 17–19)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(p, q) = 0}$</span>‏ (تحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p, q}$</span>‏ فقط، بلا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x, y, z}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p = a, q = b}$</span>‏، ونوجد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${b = F_1(a)}$</span>‏ من المعادلة، ونعوض في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z = ax + F_1(a)y + c}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>Form 2 (Clairaut — القسم 1.3.3)</b> <i>(PDF ص 19–20)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z = px + qy + f(p, q)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏نضع مباشرةً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p = a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${q = b}$</span>‏ فيكون الحل التام: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z = ax + by + f(a, b)}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>Form 3 (القسم 1.3.4)</b> <i>(PDF ص 21–23)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(z, p, q) = 0}$</span>‏ (تحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z, p, q}$</span>‏ فقط، بلا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x, y}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏نفرض ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${u = x + ay}$</span>‏ ونستبدل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p = \frac{dz}{du}}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${q = a\frac{dz}{du}}$</span>‏ ونحل المعادلة التفاضلية العادية

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>Form 4 (Separable — القسم 1.3.5)</b> <i>(PDF ص 23–25)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f_1(x, p) = f_2(y, q)}$</span>‏ (لا يوجد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${z}$</span>‏، ويمكن فصل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${x,p}$</span>‏ عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${y,q}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏نساوي الطرفين بثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${a}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f_1(x,p)=a, f_2(y,q)=a}$</span>‏، نوجد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p(x), q(y)}$</span>‏ ونكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${dz = p\,dx + q\,dy}$</span>‏

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏<b>طريقة شاربيت العامة (القسم 1.3.6)</b> <i>(PDF ص 25–27)</i>

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${f(x, y, z, p, q) = 0}$</span>‏ (جميع المتغيرات مختلطة ولا تنطبق الأشكال 1–4)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏نكتب كسور شاربيت الخمسة، نوجد علاقة ثانية بين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${p, q}$</span>‏ بثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${a}$</span>‏، ثم نكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${dz = p\,dx + q\,dy}$</span>‏

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏📖 تفصيل الأشكال الأربعة مع أمثلة المذكرة المحلولة خطوة بخطوة:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏1️⃣ الشكل الأول ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(p, q) = 0}$</span>‏ (القسم 1.3.2 — ص 16–18 بالمذكرة / PDF ص 17–19):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>لماذا ينجح؟</b> إذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = b}$</span>‏ ثابتين، فإن التفاضل الكلي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dz = p\,dx + q\,dy = a\,dx + b\,dy}$</span>‏ يعطي بالتكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = ax + by + c}$</span>‏. وبما أن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a, b) = 0}$</span>‏، نحسب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ بدلالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ ليبقى ثابتان مستقلان فقط ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, c)}$</span>‏!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 1 (PDF ص 18):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p^2 + p = q^2}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = a, q = b \implies a^2 + a = b^2 \implies b = \pm\sqrt{a^2 + a}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ إذن التكامل التام هو: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = ax \pm \sqrt{a^2 + a}\,y + c}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 2 (PDF ص 18–19):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${pq = k}$</span>‏ (حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k}$</span>‏ ثابت معطى):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = a, q = b \implies ab = k \implies b = \frac{k}{a}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ إذن التكامل التام هو: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = ax + \frac{k}{a}y + c}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏2️⃣ الشكل الثاني: معادلة كليروت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = px + qy + f(p, q)}$</span>‏ (القسم 1.3.3 — ص 18–19 بالمذكرة / PDF ص 19–20):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>قاعدة الحل:</b> ضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = b}$</span>‏ مباشرةً في المعادلة! <b>(تنبيه: لا تضف ثابتاً ثالثاً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${+c}$</span>‏ لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ مستقلان عن بعضهما هنا!)</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 2 (PDF ص 20 — انتبه لخدعة إعادة الترتيب!):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} = z - 5\frac{\partial z}{\partial x} + \frac{\partial z}{\partial x}\frac{\partial z}{\partial y}}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نكتبها برموز مونج: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${xp + yq = z - 5p + pq}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>نفصل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ في طرف بمفرده:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = xp + yq + 5p - pq}$</span>‏ (الآن أصبحت على صيغة كليروت تماماً حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(p,q) = 5p - pq}$</span>‏!).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = a, q = b}$</span>‏ فنحصل على التكامل التام: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{z = ax + by + 5a - ab}}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏3️⃣ الشكل الثالث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(z, p, q) = 0}$</span>‏ (القسم 1.3.4 — ص 20–22 بالمذكرة / PDF ص 21–23):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما تغيب ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ صراحةً عن المعادلة وتظهر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z, p, q}$</span>‏ فقط، نتبع <b>الخطوات الأربع في المذكرة (PDF ص 21)</b>:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> نفرض المتغير المجمع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u = x + ay}$</span>‏ (حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ ثابت اختياري).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> باستخدام قاعدة السلسلة (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial u}{\partial x} = 1}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{\partial u}{\partial y} = a}$</span>‏) نستبدل:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p = \frac{dz}{du}, \qquad q = a\frac{dz}{du}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> نعوض في المعادلة الأصلية فتتحول إلى معادلة تفاضلية عادية (ODE) من الرتبة الأولى في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u}$</span>‏، فنفصل المتغيرات ونكاملها (مع إضافة ثابت التكامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c}$</span>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>4.</b> نعيد استبدال ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u}$</span>‏ بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x + ay}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 1 (PDF ص 21–22):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = p + q}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u = x + ay}$</span>‏ ونعوض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = \frac{dz}{du}}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = a\frac{dz}{du}}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = \frac{dz}{du} + a\frac{dz}{du} = (1 + a)\frac{dz}{du} \implies du = (1 + a)\frac{dz}{z}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ بالتكامل: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u + c = (1 + a)\ln z \implies x + ay + c = \ln\left(z^{1+a}\right) \implies \boxed{z = e^{\frac{x + ay + c}{1 + a}}}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 2 (PDF ص 22–23):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p^2 z - q^2 = 1}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نعوض بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = \frac{dz}{du}, q = a\frac{dz}{du}}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\left(\frac{dz}{du}\right)^2 z - a^2\left(\frac{dz}{du}\right)^2 = 1 \implies (z - a^2)\left(\frac{dz}{du}\right)^2 = 1 \implies \pm\sqrt{z - a^2}\,dz = du$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ بالتكامل والتعويض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${u = x + ay}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{\pm\frac{2}{3}(z - a^2)^{3/2} = x + ay + c}}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏4️⃣ الشكل الرابع القابل للفصل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_1(x, p) = f_2(y, q)}$</span>‏ (القسم 1.3.5 — ص 22–24 بالمذكرة / PDF ص 23–25):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما لا يظهر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ في المعادلة، ونستطيع عزل كل ما يخص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x, p)}$</span>‏ في طرف وكل ما يخص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(y, q)}$</span>‏ في الطرف الآخر:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> نساوي الطرفين بثابت اختياري ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_1(x, p) = f_2(y, q) = a}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> نحل المعادلة الأولى لإيجاد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = g_1(x, a)}$</span>‏، ونحل الثانية لإيجاد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = g_2(y, a)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> نعوض في التفاضل الكلي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dz = p\,dx + q\,dy}$</span>‏ ونكامل الطرفين:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z = \int g_1(x, a)\,dx + \int g_2(y, a)\,dy + b}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>مثال 1 (PDF ص 23–24):</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = 2xq^2}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ بقسمة الطرفين على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ نفصل المتغيرات: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{p}{x} = 2q^2 = a}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ إذن: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{p}{x} = a \implies p = ax}$</span>‏، و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${2q^2 = a \implies q = \pm\sqrt{\frac{a}{2}}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نعوض في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dz = p\,dx + q\,dy = ax\,dx \pm \sqrt{\frac{a}{2}}\,dy \xrightarrow{\int} \boxed{z = \frac{a}{2}x^2 \pm \sqrt{\frac{a}{2}}\,y + b}}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🏛️ خامساً: طريقة شاربيت العامة (Charpit's Method — القسم 1.3.6، ص 24–26 / PDF ص 25–27)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما تحتوي المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y, z, p, q) = 0}$</span>‏ على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z}$</span>‏ مع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x, y}$</span>‏ ولا تنطبق عليها الأشكال الأربعة السابقة، نستخدم <b>معادلات شاربيت المساعدة (PDF ص 25)</b>:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dp}{f_x + p f_z} = \frac{dq}{f_y + q f_z} = \frac{dz}{-p f_p - q f_q} = \frac{dx}{-f_p} = \frac{dy}{-f_q}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏📖 المثال المحلول الكامل بطريقة شاربيت (Example 2 — ص 25–26 بالمذكرة / PDF ص 26–27):

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> حل المعادلة التفاضلية الجزئية غير الخطية بطريقة شاربيت:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$2zx - px^2 - 2qxy + pq = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 1:</b> نصفر الطرف الأيمن ونعرف الدالة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$f(x, y, z, p, q) = 2zx - px^2 - 2qxy + pq = 0 \qquad \cdots (5)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 2:</b> نحسب المشتقات الجزئية الخمس للدالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$f_x = 2z - 2px - 2qy, \qquad f_y = -2qx, \qquad f_z = 2x, \qquad f_p = -x^2 + q, \qquad f_q = -2xy + p$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏⚠️ <b>تنبيه خطأ مطبعي في مذكرة الكلية (PDF صفحة 27، السطر الأول):</b><br>
‏في أعلى صفحة 27 من ملف الـ PDF، التصقت معادلتا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_z}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_p}$</span>‏ ببعضهما بدون فاصلة فظهرتا هكذا: ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_z = 2x f_p = -x^2 + q}$</span>‏</code>‏! والصواب أنهما مشتقتان منفصلتان: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_z = 2x}$</span>‏ ، و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f_p = -x^2 + q}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 3:</b> نعوض في معادلات شاربيت المساعدة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dp}{(2z - 2px - 2qy) + p(2x)} = \frac{dq}{-2qx + q(2x)} = \frac{dz}{-p(-x^2+q) - q(-2xy+p)} = \frac{dx}{x^2 - q} = \frac{dy}{2xy - p}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 4 (اختيار أسهل كسر):</b> انظر إلى مقام الكسر الثاني تحت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dq}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$f_y + q f_z = -2qx + 2qx = 0 \implies \frac{dq}{0} \implies dq = 0 \implies \boxed{q = c} \quad (\text{حيث } c \text{ ثابت اختياري!})$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 5 (إيجاد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ بالتعويض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = c}$</span>‏ في المعادلة الأصلية 5):</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$2zx - px^2 - 2cxy + cp = 0 \implies p(x^2 - c) = 2x(z - cy) \implies \boxed{p = \frac{2x(z - cy)}{x^2 - c}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>الخطوة 6 (التعويض في التفاضل الكلي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dz = p\,dx + q\,dy}$</span>‏ والتكامل):</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$dz = \frac{2x(z - cy)}{x^2 - c}dx + c\,dy \implies dz - c\,dy = \frac{2x(z - cy)}{x^2 - c}dx$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لاحظ أن البسط على اليسار ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${dz - c\,dy}$</span>‏ هو تفاضل المقدار ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(z - cy)}$</span>‏ لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c}$</span>‏ ثابت! نفصل المتغيرات:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{d(z - cy)}{z - cy} = \frac{2x\,dx}{x^2 - c} \xrightarrow{\text{بتكامل الطرفين}} \ln|z - cy| = \ln|x^2 - c| + \ln b$$

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z - cy = b(x^2 - c) \implies \boxed{z = b(x^2 - c) + cy}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏وهو التكامل التام المحتوي على الثابتين الاختياريين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c}$</span>‏!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏3. 📝 تحدي الورقة والقلم 1 (صيغة كليروت — المثال 1 ص 19 / PDF ص 20)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> أوجد التكامل التام لمعادلة كليروت التالية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = p x + q y + p q$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>خطوات الحل على ورقتك:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> تحقق أن المعادلة على صيغة كليروت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = px + qy + f(p, q)}$</span>‏ حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(p, q) = pq}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> عوض عن الميلين بالثابتين الاختياريين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = a}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = b}$</span>‏.

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏استبدل كل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a}$</span>‏ وكل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏ بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b}$</span>‏ في المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = px + qy + pq}$</span>‏. لا تضف أي ثوابت إضافية لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, b)}$</span>‏ هما الثابتان الاختياريان بالفعل.

</div>

</details>

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_clairaut = "z = ax + by + ab"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab3.check_paper_challenge_1(student_clairaut)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏4. 🚀 تحدي جسر مركبة المريخ (من المستويات إلى مخروط المسافة الإقليدية)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نريد الآن حل معادلة المسافة (Eikonal Equation) التي يعتمد عليها رادار المركبة وتوسيع العوائق:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$p^2 + q^2 = 1$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>كيف يرتبط التكامل التام (Form 1) بمخروط المسافة الدائري حول الصخرة؟</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> المعادلة من <b>الشكل الأول (Form 1: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(p,q)=0}$</span>‏)</b> لأنها تحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q}$</span>‏ فقط.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> إذا وضعنا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${p = \cos\alpha}$</span>‏، فإن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${q = \sin\alpha}$</span>‏ (لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\cos^2\alpha + \sin^2\alpha = 1}$</span>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> إذن عائلة المستويات المارة بنقطة الأصل هي:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = x\cos\alpha + y\sin\alpha$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏كل قيمة للزاوية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\alpha}$</span>‏ تمثل موجة مستوية تسير في اتجاه الزاوية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\alpha}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>4.</b> لإيجاد <b>الغلاف الهندسي (Envelope)</b> لجميع الاتجاهات الممكنة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\alpha \in [0, 2\pi)}$</span>‏، نشتق بالنسبة للبارامتر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\alpha}$</span>‏ ونساوي المشتقة بالصفر:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{\partial z}{\partial \alpha} = -x\sin\alpha + y\cos\alpha = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>5.</b> بتربيع المعادلتين وجمعهما للتخلص من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\alpha}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z^2 + 0^2 = (x\cos\alpha + y\sin\alpha)^2 + (-x\sin\alpha + y\cos\alpha)^2 = x^2(\cos^2\alpha + \sin^2\alpha) + y^2(\sin^2\alpha + \cos^2\alpha)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أكمل تبسيط الطرف الأيمن واكتب معادلة سطح المسافة الناتج بدلالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x, y, z}$</span>‏:

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1 (المتطابقة المثلثية)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏بما أن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\cos^2\alpha + \sin^2\alpha = 1}$</span>‏، يتبسط الطرف الأيمن إلى مجموع مربعي الإحداثيين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏.

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2 (الصيغة النهائية)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏المعادلة تربط بين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x^2 + y^2}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z^2}$</span>‏، وتمثل مخروط المسافة الذي مقطعه الأفقي عند أي مسافة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = R}$</span>‏ هو دائرة نصف قطرها ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${R}$</span>‏.

</div>

</details>

</div>


In [ ]:
# خلية تقييم: مخروط المسافة الإقليدية
### START CODE HERE ###
student_envelope = "x^2 + y^2 = z^2"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab3.check_paper_challenge_2(student_envelope)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏5. 💻 التطبيق البرمجي ببايثون: بناء خريطة المسافات وتوسيع العوائق (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate</code>‏)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏من المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x^2 + y^2 = z^2}$</span>‏ (أو بالنسبة لمصدر عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(y_0, x_0)}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(x - x_0)^2 + (y - y_0)^2 = T^2}$</span>‏)، نحصل على دالة المسافة الإقليدية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$T(x, y) = \sqrt{(x - x_0)^2 + (y - y_0)^2}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في بايثون، الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">np.hypot(dx, dy)</code>‏ تحسب الجذر التربيعي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sqrt{\Delta x^2 + \Delta y^2}}$</span>‏ بدقة عددية عالية، وهي نفس القاعدة الهندسية التي تستخدمها دالة <a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate(raw, radius)</code>‏</a> لإنشاء قرص الأمان الدائري حول كل صخرة!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أكمل الدالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">compute&#95;eikonal&#95;wavefront</code>‏ أدناه:

</div>

</div>


In [ ]:
# دالة تقييم برمجية: compute_eikonal_wavefront
def compute_eikonal_wavefront(grid_size=(50, 50), source=(25, 25)):
    # تحسب خريطة المسافة الإقليدية T(x, y) التي تحقق معادلة Eikonal: p^2 + q^2 = 1
    # ملاحظة: source = (row_y, col_x)
    Ny, Nx = grid_size
    Y, X = np.ogrid[:Ny, :Nx]
    
    ### START CODE HERE ###
    dist = np.hypot(X - source[1], Y - source[0])
    ### END CODE HERE ###
    
    return dist

# تشغيل الاختبار الآلي:
test_lab3.test_eikonal_wavefront_distance(compute_eikonal_wavefront)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏6. 🎨 الرؤية البصرية والربط المباشر بملف ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/terrain.py</code>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لرؤية:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>على اليسار:</b> خريطة المسافة الإقليدية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${T(x, y)}$</span>‏ التي تحقق ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\Vert \nabla T\Vert ^2 = p^2 + q^2 = 1}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>على اليمين:</b> التطبيق الفعلي في المشروع عبر استدعاء دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">terrain.dilate()</code>‏ من <a href="rover_path_finder/terrain.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/terrain.py</code>‏</a> لإضافة هامش أمان دائري حول الحفر والصخور قبل تخطيط مسار المركبة!

</div>

</div>


In [ ]:
from rover_path_finder import terrain

wavefront = compute_eikonal_wavefront((60, 60), source=(30, 30))
t_data = terrain.make_terrain(N=100, seed=42, rover_radius=3)
elev, raw_hazard, buffered_hazard = t_data["h"], t_data["raw"], t_data["hazard"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

cs = axes[0].contourf(wavefront, levels=20, cmap='plasma')
plt.colorbar(cs, ax=axes[0], label="Euclidean Distance T(x, y)")
axes[0].plot(30, 30, 'wo', markersize=9, label="Source (x0, y0)")
axes[0].set_title(r"1. Eikonal Distance Field: $\|\nabla T\|^2 = p^2 + q^2 = 1$", fontsize=12)
axes[0].legend()

# عرض العوائق الأصلية (بالأحمر الداكن) وهامش الأمان الإقليدي المضاف (بالبرتقالي)
safety_margin_only = buffered_hazard & (~raw_hazard)
overlay = np.zeros((100, 100, 3))
overlay[ ~buffered_hazard ] = [0.95, 0.95, 0.95]   # أرض آمنة
overlay[ safety_margin_only ] = [1.0, 0.65, 0.1]   # هامش الأمان الدائري (r <= 3)
overlay[ raw_hazard ] = [0.75, 0.1, 0.1]           # الصخور والحفر الأصلية

axes[1].imshow(overlay, origin='upper')
axes[1].set_title("2. Real Engine: terrain.dilate(raw, rover_radius=3)", fontsize=12)
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")

plt.tight_layout()
plt.show()

print("📌 ملخص اللاب 3:")
print("  1) المعادلات غير الخطية من الرتبة الأولى تُحل بالأشكال القياسية الأربعة أو بطريقة شاربيت العامة.")
print("  2) معادلة المسافة p^2 + q^2 = 1 تنتج مخروط المسافة الدائري x^2 + y^2 = z^2.")
print("  3) دالة terrain.dilate() في المشروع تستخدم هذه المسافة الإقليدية لتوسيع العوائق بنصف قطر المركبة.")
